In [2]:
import sqlite3
import pandas as pd
from src.query import query_table
import numpy as np
from src.score import Score
import os 
import os
from termcolor import colored

from src.score import Score
from src.query import npast_games_words

import datetime

import time 



current_dir = os.getcwd()
con = sqlite3.connect("data/main_database.sqlite")
# con = sqlite3.connect("data/main_database.db")
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', lambda x: '%.3f' % x)

In [2]:
df = pd.read_csv('game_settings.csv').drop('Unnamed: 0', axis=1)
df 
pd.concat([df['game_id'], pd.json_normalize(df['game_settings'])], axis=1)

,game_id
0,1179895951


In [4]:
pd.json_normalize(df['game_settings'].apply(eval))

,word_count,word_length_min,word_length_max,capitalized_words,capitalized_letters,punctuation,punctuation_char,force_shift,difficulty,n_games_banned_words,train,train_easy,player_name,display_chart,display_wpm,display_words_left,display_word_count,sentence
0,2,4,1000,1,0,0,0,0,Medium,0,False,True,NPC,False,False,True,-1,suggesting driven


In [5]:
df = query_table(con, 'clean_games_settings')
df = df.drop(['game_id'], axis=1)
df.to_sql('clean_games_settings', con, if_exists='append', index=False)

13

In [6]:
query_table(con, 'games_summary', 'AND word_count = 2  AND word_length_max = 1000  AND word_length_min = 4  AND capitalized_words = 1  AND capitalized_letters = 0  AND punctuation = 0  AND force_shift = 0  AND difficulty = "Medium"  AND train = False  AND train_easy = True')

,game_id,maxdatetime_unix,mindatetime_unix,date_time,game_duration,keys_to_press,keys_pressed,errors,accuracy,wpm,cps,sentence_length,score,sentence,word_count,word_length_max,word_length_min,capitalized_words,capitalized_letters,punctuation,force_shift,difficulty,train,train_easy,player_name


In [7]:
query = '''


with kp as (
select 
    distinct
    key
    , time
    , correct_key
    , game_id
    , row_number() over(partition by key order by time desc) as key_index
    

from keys_pressed 
where 1=1
    and correct_key = 1
    and game_id > 100
    and time is not null
order by time
)

, tbl1 as (
select
    key
    , lead(key) over(partition by game_id order by game_id, time) following_key
    , time
    , lead(time) over(partition by game_id order by game_id, time) following_time
    , game_id

from kp
where 1=1
    and key_index < 11
)


select 
    following_key
    , avg(following_time - time) as time_diff
    , count(*) count

from tbl1
where 1=1
    and following_key is not null
group by 1
having 1=1
    --and count < 200
order by time_diff asc




'''

df = pd.read_sql_query(query, con)#.sort_values('time_diff', ascending=False).query('following_key.str.lower().str.isalpha() and following_key.str.lower() == following_key and following_key.str.len() == 1')
df

,following_key,time_diff,count
0,h,0.089,3
1,u,0.107,5
2,e,0.111,10
3,,0.130,10
4,n,0.134,10
5,v,0.136,2
6,a,0.137,3
7,o,0.152,2
8,p,0.178,5
9,f,0.181,1


In [8]:
query = '''
with kp as (
select 
    distinct
    key
    , time
    , correct_key
    , game_id
    , row_number() over(partition by key order by time desc) as key_index
    

from keys_pressed 
where 1=1
    and correct_key = 1
    and game_id > 100
    and time is not null
order by time
)

, tbl1 as (
select
    key
    , lead(key) over(partition by game_id order by game_id, time) following_key
    , time
    , lead(time) over(partition by game_id order by game_id, time) following_time
    , game_id

from kp
where 1=1
    and key_index <= 200
)

select 
    following_key
    , avg(following_time - time) as time_diff
    , count(*) count

from tbl1
where 1=1
    and following_key is not null
group by 1
having 1=1
    --and count < 200
order by time_diff asc
'''

df = pd.read_sql_query(query, con).sort_values('time_diff', ascending=True).query('following_key.str.lower().str.isalpha() and following_key.str.lower() == following_key and following_key.str.len() == 1')
df.head(50)

,following_key,time_diff,count
0,v,0.079,2
1,h,0.089,3
2,u,0.106,6
3,n,0.116,14
4,g,0.127,13
6,p,0.135,5
7,a,0.137,3
8,e,0.154,17
9,t,0.170,10
10,i,0.178,17


In [9]:
query = '''
wITH last_200 AS (
  SELECT 
    key,
    time,
    LAG(time) OVER (PARTITION BY key ORDER BY time) AS prev_timestamp
  FROM keys_pressed
  WHERE correct_key = 1
  AND game_id IN (
    SELECT game_id
    FROM (
      SELECT 
        game_id,
        ROW_NUMBER() OVER (PARTITION BY key ORDER BY time DESC) AS row_num
      FROM keys_pressed
      WHERE correct_key = 1
    ) temp
    WHERE row_num <= 200
  )
)
SELECT 
  key,
  AVG(time - prev_timestamp) AS avg_time_to_type
FROM last_200
WHERE prev_timestamp IS NOT NULL
GROUP BY key;
'''

df = pd.read_sql_query(query, con).sort_values('avg_time_to_type', ascending=True)##.query('following_key.str.lower().str.isalpha() and following_key.str.lower() == following_key and following_key.str.len() == 1')
df.head(50)

,key,avg_time_to_type
3,c,1.542
10,l,86.075
5,e,191.693
8,i,191.708
9,k,200.626
16,s,203.873
12,n,235.946
6,g,236.017
1,a,236.548
13,o,273.523


In [10]:
df = pd.read_sql_query('select * from games_summary where maxdatetime_unix is not null', con)
df['date_time'] = pd.to_datetime(df['date_time'])
df.head(1)

,game_id,maxdatetime_unix,mindatetime_unix,date_time,game_duration,keys_to_press,keys_pressed,errors,accuracy,wpm,cps,sentence_length,score,sentence,word_count,word_length_max,word_length_min,capitalized_words,capitalized_letters,punctuation,force_shift,difficulty,train,train_easy,player_name
0,1179895951,1684198387.749,1684198385.639,2023-05-15 20:53:07,2.110,17,17,0,1.000,96.677,8.056,17,1643.507,suggesting driven,2,1000,4,1,0,0,0,Medium,0,0,npc


In [11]:








--  -  --------
26  n  0.135908
25     0.136445
24  o  0.14367
23  u  0.164228
22  e  0.173107
21  i  0.181243
20  p  0.183525
19  l  0.185843
18  t  0.189996
17  h  0.19383
16  r  0.195024
15  a  0.206001
14  y  0.21291
13  k  0.217505
12  m  0.219152
11  s  0.227393
10  g  0.23129
 9  d  0.240956
 8  j  0.242916
 7  f  0.253575
 6  q  0.256713
 5  c  0.259814
 4  x  0.272079
 3  w  0.276446
 2  z  0.283076
 1  b  0.288689
 0  v  0.297809

SyntaxError: invalid syntax (431508041.py, line 1)

In [ ]:
fig = go.Figure(data=go.Scatter(x=df['date_time'], y=df['wpm']))
fig.update_layout(template='plotly_dark', title='WPM over time', xaxis_title='Date', yaxis_title='WPM')
fig.show()

In [ ]:
query = '''
with kp as (
select 
    distinct
    key
    , time
    , correct_key
    , game_id
    , row_number() over(partition by key order by time) as key_index
    

from keys_pressed 
where 1=1
    and time is not null
order by time
)




, tbl1 as (
select
    key
    , lead(key) over(partition by game_id order by game_id, time) following_key
    , time
    , lead(time) over(partition by game_id order by game_id, time) following_time
    , game_id

from kp
where 1=1
    and correct_key = 1
    and game_id > 100
    and key_index < 200
)

select 
    following_key
    , avg(following_time - time) as time_diff
    , count(*) count

from tbl1
where 1=1
    and following_key is not null
group by 1
having 1=1
    -- and count > 100
order by time_diff asc


'''

df = pd.read_sql_query(query, con)
df

In [ ]:
query = '''

select 
    distinct
    key
    , time
    , correct_key
    , game_id
    , row_number() over(partition by key order by time) as key_index
    

from keys_pressed 
where 1=1
    and time is not null
order by time

'''



pd.read_sql_query(query, con)

In [ ]:
(str(datetime.timedelta(seconds=1000000)).replace(' days, ', 'd ').replace(':', 'h ').replace(':', 'm ') + 's').upper()

In [ ]:
blue_lch = [54., 70., 4.6588]
l_mid = 40.
red_lch = [54., 90., 0.35470565 + 2* np.pi]
gray_lch = [55., 0., 0.]
blue_rgb = lch2rgb(blue_lch)
red_rgb = lch2rgb(red_lch)
gray_rgb = lch2rgb(gray_lch)
white_rgb = np.array([1.,1.,1.])


light_blue_rgb = np.array([127., 196, 252])/255
light_red_rgb = np.array([255., 127, 167])/255

# define a perceptually uniform color scale using the Lch color space
reds = []
greens = []
blues = []
alphas = []
nsteps = 100
l_vals = list(np.linspace(blue_lch[0], l_mid, nsteps//2)) + list(np.linspace(l_mid, red_lch[0], nsteps//2))
c_vals = np.linspace(blue_lch[1], red_lch[1], nsteps)
h_vals = np.linspace(blue_lch[2], red_lch[2], nsteps)
for pos,l,c,h in zip(np.linspace(0, 1, nsteps), l_vals, c_vals, h_vals):
    lch = [l, c, h]
    rgb = lch2rgb(lch)
    reds.append((pos, rgb[0], rgb[0]))
    greens.append((pos, rgb[1], rgb[1]))
    blues.append((pos, rgb[2], rgb[2]))
    alphas.append((pos, 1.0, 1.0))

red_blue = LinearSegmentedColormap('red_blue', {
    "red": reds,
    "green": greens,
    "blue": blues,
    "alpha": alphas
})

Analyzing your own data as a data person is so much fun, you get to analyze your own behavior. A couple of months ago, I made a game in python to type faster and I use the data produced during the games so the next ones will have words containing the letters and combinations I struggle to type. 

I made the game as I switched keyboard layout from french to Us and also wanted to type without looking at my keyboard. I went from looking at it, typing with 7 fingers to typing with 9 (left thumb still useless af) but without looking. It was terrible at first, I was at 30 word per minute and thought it'd be forever but i'm hiting close to 80 wpm now which is nice. 

In [ ]:
for file in os.listdir('data/text/'):
    with open('data/text/' + file) as f: 
        word_list = f.read().split('\n')
        l = [len(word) for word in word_list]
        print(file, max(l), len(word_list), round(np.mean(l), 2), round(np.median(l), 2), round(np.std(l), 2))

In [ ]:
with open('data/text/google_most_used_words.txt') as file: 
    word_list = file.read().split('\n')



def only_one_letter(word_list): 
    query = '''


    with tbl1 as (
    select
        key
        , lead(key) over(partition by game_id order by game_id, time) following_key
        , time
        , lead(time) over(partition by game_id order by game_id, time) following_time
        , game_id

    from keys_pressed
    where 1=1
        and correct_key = 1
        and game_id > 100
    )

    select 
        following_key
        , avg(following_time - time) as time_diff
        , count(*) count

    from tbl1
    where 1=1
        and following_key is not null
    group by 1
    having 1=1
        -- and count > 100
    order by time_diff asc


    '''

    df = pd.read_sql_query(query, con).sort_values('time_diff', ascending=False).query('following_key.str.lower().str.isalpha() and following_key.str.lower() == following_key and following_key.str.len() == 1')
    print(df.head())
    letter = df.iloc[0, 0]
    voyelle = df.query('following_key.str.contains("a|e|i|o|u|y")').iloc[0, 0]

    letter_count = {}
    for word in word_list:
        letter_count[word] = word.count(letter) #round(np.ceil(word.count(letter) + word.count(voyelle)/5)) # /3 is a weight so it does't count as much as a letter. In testing
    
    freq = dict(sorted(letter_count.items(), key=lambda item: item[1], reverse=True))
    print(len(freq.keys()), len(set(freq.keys())))
    # print(freq)
    total_weights = sum(freq.values())
    probas = [weight/total_weights for weight in freq.values()]
    
    
    # print(list(freq.keys()))
    word_list = np.random.choice(list(freq.keys()), 25, False, probas)

    return list(word_list)



lst = only_one_letter(word_list)
sum([x.count('z') > 0 for x in lst]), len(set(lst))

In [ ]:
# import sqlite3
# con = sqlite3.connect('data/main_database.db')
# con.execute('delete from games_settings where game_id = 7645338967')

In [ ]:
import os 
import pandas as pd
import sqlite3
from tqdm import tqdm

allowed_extensions = ['db', 'sqlite', 'sqlite3']
parent_path = 'data/databases_archives/'
data = {}
for db_name in tqdm(os.listdir(parent_path)): 
    df_sqlite_master = pd.DataFrame()
    if db_name.split('.')[-1] not in allowed_extensions:
        continue 

    db_path = parent_path + db_name
    con = sqlite3.connect(db_path)
    df = pd.read_sql_query('select * from sqlite_master', con)
    df_sqlite_master = pd.concat([df_sqlite_master, df], axis=0)

    for table_name in df_sqlite_master['name']: 
        if table_name not in data.keys(): 
            data[table_name] = pd.read_sql_query('select * from ' + table_name, con)
        else: 
            data[table_name] = pd.concat([data[table_name], pd.read_sql_query('select * from ' + table_name, con)], axis=0)


con_archive = sqlite3.connect('data/archive_database.db')
for table_name in data.keys():
    data[table_name].dr
    print(table_name, data[table_name].shape)

In [ ]:
new_query = '''
with tbl as (
select 
    key
    , time
    , lead(time) over(partition by game_id order by time desc) lead_time
    , time - lead(time) over(partition by game_id order by time desc) time_diff
    --, lead(key) over(order by time desc) lead_key
    --, lead(time) over(order by time desc) lead_time
    , row_number() over(partition by key order by time desc) rank

from keys_pressed
left join clean_games_settings using(game_id)
where 1=1 
    and word_count > 20
    and correct_key = 1

order by time 
)


select 
    key
    , avg(time_diff) time_diff

from tbl 
where rank <= 200

group by 1
order by 2 desc

'''

df = pd.read_sql_query(new_query, con_archive)
df

In [ ]:
new_query = '''
with tbl as (
select 
    key
    , time
    , lead(time) over(partition by game_id order by time desc) lead_time
    , time - lead(time) over(partition by game_id order by time desc) time_diff
    --, lead(key) over(order by time desc) lead_key
    --, lead(time) over(order by time desc) lead_time
    , row_number() over(partition by key order by time desc) rank

from keys_pressed
left join clean_games_settings using(game_id)
where 1=1 
    and word_count > 20
    and correct_key = 1

order by time 
)


select 
    key
    , avg(time_diff) time_diff

from tbl 
where rank <= 200

group by 1
order by 2 desc

'''
letters = 'qwertyuiopasdfghjklzxcvbnm')
df = pd.read_sql_query(new_query, con).sort_values('time_diff', ascending=True)#.query('key.isin(@letters)')

letters



In [ ]:
query = '''


with tbl1 as (
select
    key
    , lead(key) over(partition by game_id order by game_id, time) following_key
    , time
    , lead(time) over(partition by game_id order by game_id, time) following_time
    , game_id

from keys_pressed
where 1=1
    and correct_key = 1
    and game_id > 100
)


, final as (
select 
    following_key
    --, avg(following_time - time) as time_diff
    --, count(*) count
    , following_time
    , time
    , rank() over(partition by following_key order by following_time desc) rank

from tbl1
where 1=1
    and following_key is not null
)    

select 
    following_key
    , avg(following_time - time) as time_diff
    , count(*) count

from final
where 1=1
    --and rank < 100
order by time_diff asc


'''
query = '''


        with tbl1 as (
        select
            key
            , lead(key) over(partition by game_id order by game_id, time) following_key
            , time
            , lead(time) over(partition by game_id order by game_id, time) following_time
            , game_id

        from keys_pressed
        where 1=1
            and correct_key = 1
            and game_id > 100
        )

        select 
            following_key
            , avg(following_time - time) as time_diff
            , count(*) count

        from tbl1
        where 1=1
            and following_key is not null
        group by 1
        having 1=1
            -- and count > 100
        order by time_diff asc


        '''


# df = pd.read_sql_query(query, con)#.sort_values('time_diff', ascending=False).query('following_key.str.lower().str.isalpha() and following_key.str.lower() == following_key and following_key.str.len() == 1')
df = pd.read_sql_query(query, con).query('following_key.str.lower().str.isalpha() and following_key.str.lower() == following_key and following_key.str.len() == 1')
# print(df.head(20))
# letter = df.iloc[0, 0]
# voyelle = df.query('following_key.str.contains("a|e|i|o|u|y")').iloc[0, 0]

# letter_count = {}
# for word in words:
#     letter_count[word] = word.count(letter) + word.count(voyelle)/3

# freq = dict(sorted(letter_count.items(), key=lambda item: item[1], reverse=True))
# word_list = [word for word, count in freq.items() for x in range(count)]
# word_list
df

In [ ]:
len(words)

In [ ]:
data = {}

for root, dirs, files in os.walk(os.getcwd()):
    for filename in files:
        file_path = os.path.join(root, filename)

        if file_path.endswith('.db') or file_path.endswith('.sqlite'):
            current_con = sqlite3.connect(file_path)
            try: 
                db_tables = pd.read_sql_query('select * from sqlite_master', current_con)['tbl_name'].tolist()
            except pd.errors.DatabaseError: 
                continue
            # print(db_tables)

            for table in db_tables:
                if table not in ['keys_pressed', 'games_settings']:
                    continue
                if table not in data.keys():
                    data[table] = []

                current_df = pd.read_sql_query(f'select * from {table}', current_con)
                # print(current_df.shape)
                data[table].append(current_df)


df_keys_pressed = pd.concat(data['keys_pressed'], axis=0).drop_duplicates()[['key', 'time', 'correct_key', 'shift', 'game_id']]
df_games_settings = pd.concat(data['games_settings'], axis=0).drop_duplicates().query('game_settings.str[0] == "{"')[['game_id', 'game_settings']]

df_keys_pressed.shape

In [ ]:
(df_games_settings['game_settings'].str[0] == '{').value_counts()

In [ ]:
game_data = [['l', True, None, 1675125990.710492, 1209525288], ['a', True, None, 1675125990.9652321, 1209525288], ['w', True, None, 1675125991.324478, 1209525288], ['n', True, None, 1675125991.510137, 1209525288], ['space', True, None, 1675125991.690339, 1209525288], ['r', True, None, 1675125991.9562728, 1209525288], ['e', True, None, 1675125992.024321, 1209525288], ['m', True, None, 1675125992.146587, 1209525288], ['o', True, None, 1675125992.2910259, 1209525288], ['t', True, None, 1675125992.377158, 1209525288], ['e', True, None, 1675125992.491313, 1209525288], ['space', True, None, 1675125992.5899289, 1209525288], ['d', True, None, 1675125992.858459, 1209525288], ['e', True, None, 1675125993.017688, 1209525288], ['v', True, None, 1675125993.2235851, 1209525288], ['i', True, None, 1675125993.345284, 1209525288], ['c', True, None, 1675125993.521666, 1209525288], ['e', True, None, 1675125993.767472, 1209525288], ['space', True, None, 1675125993.844388, 1209525288], ['c', True, None, 1675125994.023753, 1209525288], ['r', True, None, 1675125994.249588, 1209525288], ['u', True, None, 1675125994.36858, 1209525288], ['c', True, None, 1675125994.590071, 1209525288], ['i', True, None, 1675125994.691125, 1209525288], ['a', True, None, 1675125994.841098, 1209525288], ['l', True, None, 1675125994.943382, 1209525288], ['space', True, None, 1675125995.0336902, 1209525288], ['v', True, None, 1675125995.30562, 1209525288], ['e', True, None, 1675125995.483807, 1209525288], ['g', True, None, 1675125995.66958, 1209525288], ['e', True, None, 1675125995.825585, 1209525288], ['t', True, None, 1675125996.01058, 1209525288], ['a', True, None, 1675125996.18667, 1209525288], ['b', True, None, 1675125996.282815, 1209525288], ['l', True, None, 1675125996.496135, 1209525288], ['e', True, None, 1675125996.6256661, 1209525288]]


first_timestamp, last_timestamp = game_data[0][3]
game_duration = last_timestamp - first_timestamp
keys_pressed = len(game_data)
keys_to_press = list(filter(lambda x: x[1] == True))
accuracy = keys_to_press / keys_to_press
wpm = keys_to_press / (game_duration / 60) / 5


In [ ]:
game_data = {'word_count': 5, 'word_length_min': 0, 'word_length_max': 1000, 'capitalized_words_count': 0, 'capitalized_letters_count_perc': 0, 'punctuation_word_count_perc': 0, 'force_shift': 0, 'hard_mode': 'Medium', 'train_letters': 0, 'train_letters_easy_mode': 0, 'player_name': 'marc', 'sentence': 'secretary collect enormous less cost'}


first_second, last_second = df.iloc[[0, -1], 2]
game_duration = round(last_second - first_second) if last_second - first_second > 10 else round(last_second - first_second, 2) # Don't round the seconds unless games of less than 10 second (for development only unless you're putting some lame ass rules)
keys_pressed = df.shape[0]
keys_to_press = df.query('correct_key == 1').shape[0]
accuracy = keys_to_press / keys_pressed
wpm = keys_to_press / (game_duration / 60) / 5
score = accuracy * wpm * len(self.sentence)

In [ ]:
df_game_data['game_settings'][0]

In [ ]:
pd.Series({'word_count': 5, 'word_length_min': 0, 'word_length_max': 1000, 'capitalized_words_count': 0, 'capitalized_letters_count_perc': 0, 'punctuation_word_count_perc': 0, 'force_shift': 0, 'hard_mode': 'Medium', 'train_letters': 0, 'train_letters_easy_mode': 0, 'player_name': 'marc', 'sentence': 'secretary collect enormous less cost'})

In [ ]:
import toml

def read_config(config='game_default') -> dict: 
    '''Given a config file name, reads it and return a dictionary
    with the game settings.'''
    with open(f'configs/{config}.toml', 'r') as f:
        config = toml.load(f)
    
    return config


read_config()['game_config']

In [ ]:
from tabulate import tabulate
from termcolor import colored

print(tabulate([['hi', colored('oui', 'red')]], headers=['hi', 'oui']))

In [ ]:
from tabulate import tabulate
print(tabulate([['Alice', 24], ['Bob', 19]], headers=['Name', 'Age']))

### class sentence

In [ ]:
capitalized_words_count = 0 # Set a float between 0 and 1 for the percentage of word that will be generated with a/multiple random case letter
capitalized_letters_count_perc = 0 # Set a float between 0 and 1 for the percentage of the letters of the word that will be capitalized. Set an integer for the nmumber or random case statement letters. 1 is all letters capitalized not 1 word. if 'first' then only first letter will be capitalized
punctuation_word_count_perc = 0 # Same as above but for punctuation around the word






class Sentence():
    def __init__(self, )

### Current game vs best and avereage

In [ ]:
class Score():
    def __init__(self): 
        # self.game_settings = game_settings
        # self.game_id = game_settings['game_id']
        # self.word_count = game_settings['word_count']
        # self.min_word_length = game_settings['min_word_length']
        # self.max_word_length = game_settings['max_word_length']
        # self.capitalized_words_count = game_settings['capitalized_words_count']
        # self.capitalized_letters_count_perc = game_settings['capitalized_letters_count_perc']
        # self.punctuation_word_count_perc = game_settings['punctuation_word_count_perc']
        # self.force_shift = game_settings['force_shift']
        # self.hard_mode = game_settings['hard_mode']
        # self.train_letters = game_settings['train_letters']
        # self.player_name = game_settings['player_name']

        self.game_settings_query_condition = []
        # for key, value in self.game_settings.items():
        #     if key in ('game_id', 'max_word_length', 'min_word_length', 'player_name', 'sentence'):
        #         continue
        #     if type(value) == str: 
        #         condition = f'{key} == "{value}"'
        #     else:
        #         condition = f'{key} == {value}'
                
        #     self.game_settings_query_condition.append(condition)

    
    
    def best_game(self, sort_by='score', conditions=['1=1']):
        '''Return the game settings of the best game based on the sorting'''

        full_condition = ''.join([' AND ' + condition for condition in conditions + self.game_settings_query_condition])

        query = f"""
            SELECT 
                * 
            FROM summary_per_game
            WHERE 1=1
                AND keys_to_press > 100
                {full_condition}
            ORDER BY {sort_by}
            """

        df_summary = pd.read_sql_query(query, con)   
        
        # If there hasn't been any game played with the settings, catch the keyerror 
        try:
            best_game = df_summary.loc[0, :].to_dict()
        except KeyError: 
            best_game = {col: 0 for col in df_summary.columns}

        return best_game


    def max_mean_score(self, conditions=['1=1']):
        '''Return max and average metrics'''

        full_condition = ''.join([' AND ' + condition for condition in conditions + self.game_settings_query_condition])

        query = f"""
            SELECT 
                game_duration
                , sentence_length
                , accuracy
                , wpm
                , score

            FROM summary_per_game
            WHERE 1=1
                AND keys_to_press > 100
                {full_condition}
            """

        df_summary = pd.read_sql_query(query, con)   
        df_summary = df_summary.describe().loc[['max', 'mean'], :].T

        return df_summary



    def make_condition(self):
        '''Generate a string containing games settings that will act 
        as condition for the mutliple queries'''

        self.game_settings_query_condition = []
        for key, value in self.game_settings:
            if type(value) == str: 
                condition = f'{key} == "{value}"'
            else:
                condition = f'{key} == {value}'
                
            self.game_settings_query_condition.append(condition)

def get_correct_size_string(string: str, lentgh: int) -> int: 
    if len(string) > lentgh: 
        raise ValueError('String longer than the integer')

    else:
        return string + ' ' * (lentgh - len(string))


def compare_game()
    df_summary = Score().max_mean_score()
    data = df_summary.to_numpy().tolist()
    data_col_index = [[''] + list(df_summary.columns)] + [[df_summary.index[index]] + [value for value in row] for index, row in enumerate(data)]

    text_to_print = ''
    for index1, row in enumerate(data_col_index):
        for index2, value in enumerate(row):
            if type(value) in [int, float]:
                value = str(round(value, 3))
                
            text_to_print += get_correct_size_string(value, 19) + '\t'

        text_to_print += '\n'

    print(text_to_print)

### playground

In [ ]:
query = f"""
        select
            distinct
            kp.game_id
            , max(kp.time) maxdatetime_unix
            , min(kp.time) mindatetime_unix
            , max(kp.time) - min(kp.time) game_duration
            , sum(case when kp.correct_key = 1 then 1 else 0 end) keys_to_press
            , count(*) keys_pressed
            , round(CAST(sum(case when kp.correct_key = 1 then 1 else 0 end) as REAL) / count(*), 3) accuracy
            , sum(case when kp.correct_key = 1 then 1 else 0 end) / ((max(kp.time) - min(kp.time)) / 60) / 5 wpm
            , length(cgs.sentence) sentence_length
            , sum(case when kp.correct_key = 1 and key = 'space' then 1 else 0 end) + 1 word_count
            , length(cgs.sentence) * (sum(case when kp.correct_key = 1 then 1 else 0 end) / ((max(kp.time) - min(kp.time)) / 60) / 5) * round(CAST(sum(case when kp.correct_key = 1 then 1 else 0 end) as REAL) / count(*), 3) score
            , cgs.max_word_length	
            , cgs.capitalized_words_count	
            , cgs.capitalized_letters_count_perc	
            , cgs.punctuation_word_count_perc	
            , cgs.force_shift	
            , cgs.hard_mode	
            , cgs.train_letters	
            , cgs.comment	
            , cgs.player_name

        from keys_pressed kp 
        left join clean_games_settings cgs using(game_id)
        where 1=1
        group by 1
        """

df = pd.read_sql_query(query, con)
df

In [ ]:
def clean_games_settings():
    df = pd.read_sql_query('select * from games_settings', con)
    df['game_settings'] = df['game_settings'].apply(lambda x: eval(x))
    df_games_settings = pd.concat([df[['game_id']], df['game_settings'].apply(lambda x: pd.Series(x))], axis=1)
    df_games_settings.to_sql('clean_games_settings', con, if_exists='append', index=False)

In [ ]:
hard_mode = False
def load_words() -> list:
    '''Opens one of two lists of words depending on the difficulty. 
    Common_words.txt contains 3000 common words where words.txt contains 
    about 450k words generally longer and harder to type compared to the 
    common word list.
    '''

    if hard_mode == True:
        file_path = f'{current_dir}/data/words.txt'
    else: 
        file_path = f'{current_dir}/data/common_words.txt'
        
    with open(file_path) as file: 
        all_words = file.read().split('\n')

    return all_words
       

def load_query (query_name: str, text_only: bool = False) -> list:
    '''Opens a given text file name and execute the 
    query to return a pd.DataFrame
    
    Parameter
    ---------
    query_name str: name of the file name to run
    '''

    # Open file and get the query 
    with open(f'{current_dir}/data/queries/{query_name}.sql') as file: 
        query = file.read()

    # Query database
    df = pd.read_sql_query(query, con)

    if text_only == False:
        return df
    elif text_only == True:
        return query


def query_n_past_games_words(n_past_games: int) -> str:
    '''Query the past n games and returns all 
    words in used in them.
    
    parameter
    ---------
    n_past_games int: number of games to query
    '''

    query_npast_games_words = f'''
        select
            trim(sentence) sentence
            , max(time)

        from keys_pressed kp
        left join clean_games_settings gs using(game_id)
        where 1=1
            and game_id > 100
            and sentence not null
        group by 1
        order by 2 desc
        limit {n_past_games}
        '''

    df_query = pd.read_sql_query(query_npast_games_words, con)
    done_words = ' '.join(df_query['sentence']).split(' ')
    return done_words



def get_n_slowest_words(word_count: list) -> list:
    '''Among the list of word, find the words that would 
    potentially take the longest to type based on the 
    average duration it takes to the player to type all 
    each and individual letters of the words.

    parameters
    ----------
    word_count int: numbers of worst words to return
    
    '''
    # Load key score 
    df_keytime = load_query('time_per_key_pressed')
    key_score = dict(zip(df_keytime['following_key'], df_keytime['time_diff'].round(3)))

    # Get score for each word
    words = load_words()
    df_words = pd.DataFrame(words, columns=['word'])
    df_words['word_score'] = df_words['word'].apply(lambda word: sum([key_score[char] for char in word.lower() if char in key_score.keys()]))
    df_words['avg_letter_score'] = df_words['word_score'] / df_words['word'].str.len()

    # Remove words done in the past four games
    done_words = query_n_past_games_words(4)
    # Remove punctuation
    done_words = [''.join([char for char in word if char.isalpha() or char == ' ']) for word in done_words]
    df_words = df_words[df_words['word'].isin(done_words) == False]

    # Sort dataframe and pick the top 25 words with at least four letters
    top_n = df_words.sort_values('avg_letter_score', ascending=False).query('word.str.len() > 4').iloc[:word_count, 0]
    return list(top_n)


get_n_slowest_words(8)

In [ ]:
query = """
    with tbl1 as (
    select
        game_id
        , max(time) - min(time) game_duration
        , sum(case when correct_key = 1 then 1 else 0 end) keys_to_press
        , count(*) keys_pressed
        , round(CAST(sum(case when correct_key = 1 then 1 else 0 end) as REAL) / count(*), 3) accuracy
        , sum(case when correct_key = 1 then 1 else 0 end) / ((max(time) - min(time)) / 60) / 5 wpm
        , gs.game_settings

    from keys_pressed
    left join games_settings gs using(game_id)
    where 1=1
        --and game_id = 3513153090
    group by 1
    )

    select 
        *
    
    from tbl1
    where 1=1
        --wpm = (select max(wpm) from tbl1)
    """

# query = 'select * from keys_pressed left join games_settings gs using(game_id)'

df_high_score = pd.read_sql_query(query, con)
df_high_score['score'] = (df_high_score['accuracy'] * df_high_score['wpm'].round(1) * df_high_score['game_settings'].apply(lambda x: len(eval(x)['sentence']))).round(0)
df_high_score = df_high_score.sort_values('score', ascending=False).reset_index(drop=True)
best_player_name = df_high_score['game_settings'].apply(lambda x: eval(x)).loc[0]['player_name']
best_game_duration = df_high_score.loc[0, 'game_duration']
best_keys_to_press = df_high_score.loc[0, 'keys_to_press']
best_keys_pressed = df_high_score.loc[0, 'keys_pressed']
best_accuracy = df_high_score.loc[0, 'accuracy']
best_wpm = df_high_score.loc[0, 'wpm']
best_score = df_high_score.round().loc[0, 'score']
best_player_name

In [ ]:
data = [['s', True, 1664763831.267992, 5567398643], ['e', True, 1664763831.441312, 5567398643], ['n', True, 1664763831.5410728, 5567398643], ['i', True, 1664763831.707446, 5567398643], ['o', True, 1664763831.8898602, 5567398643], ['r', True, 1664763831.999372, 5567398643], ['space', True, 1664763832.115184, 5567398643], ['t', True, 1664763832.6038668, 5567398643], ['h', True, 1664763832.721718, 5567398643], ['e', True, 1664763832.8076348, 5567398643], ['y', True, 1664763833.041811, 5567398643], ['space', True, 1664763833.224024, 5567398643], ['l', True, 1664763833.388472, 5567398643], ['i', True, 1664763833.5640051, 5567398643], ['s', True, 1664763833.628949, 5567398643], ['t', True, 1664763833.7937338, 5567398643], ['e', True, 1664763833.93331, 5567398643], ['n', True, 1664763834.0548532, 5567398643], ['space', True, 1664763834.1896992, 5567398643], ['d', False, 1664763834.300885, 5567398643], ['h', False, 1664763834.4248981, 5567398643], ['a', False, 1664763834.534786, 5567398643], ['c', True, 1664763834.9380841, 5567398643], ['h', True, 1664763835.0569851, 5567398643], ['a', True, 1664763835.142268, 5567398643], ['s', True, 1664763835.394652, 5567398643], ['e', True, 1664763835.530441, 5567398643], ['space', True, 1664763835.715113, 5567398643], ['m', True, 1664763835.904495, 5567398643], ['a', True, 1664763835.964861, 5567398643], ['r', True, 1664763836.457012, 5567398643], ['k', True, 1664763836.816826, 5567398643], ['e', True, 1664763836.9502962, 5567398643], ['t', True, 1664763837.1342652, 5567398643], ['space', True, 1664763837.2958112, 5567398643], ['n', True, 1664763837.6311932, 5567398643], ['e', True, 1664763837.672338, 5567398643], ['w', True, 1664763837.8577511, 5567398643], ['s', True, 1664763838.147943, 5567398643], ['p', True, 1664763838.426924, 5567398643], ['a', True, 1664763838.552413, 5567398643], ['p', True, 1664763838.7687612, 5567398643], ['e', True, 1664763839.0281038, 5567398643], ['r', True, 1664763839.164213, 5567398643], ['space', True, 1664763839.282433, 5567398643], ['c', True, 1664763839.5093079, 5567398643], ['a', True, 1664763839.709535, 5567398643], ['b', True, 1664763839.762175, 5567398643], ['l', True, 1664763839.974815, 5567398643], ['e', True, 1664763840.060313, 5567398643], ['space', True, 1664763840.134818, 5567398643], ['h', True, 1664763840.322911, 5567398643], ['o', True, 1664763840.442032, 5567398643], ['u', True, 1664763840.637918, 5567398643], ['s', True, 1664763840.703906, 5567398643], ['e', True, 1664763840.779733, 5567398643], ['space', True, 1664763840.9210298, 5567398643], ['a', True, 1664763841.121834, 5567398643], ['t', True, 1664763841.2441862, 5567398643], ['t', True, 1664763841.4109728, 5567398643], ['e', True, 1664763841.583327, 5567398643], ['n', True, 1664763841.680005, 5567398643], ['t', True, 1664763841.772461, 5567398643], ['i', True, 1664763841.871369, 5567398643], ['o', True, 1664763842.037301, 5567398643], ['n', True, 1664763842.147409, 5567398643], ['space', True, 1664763842.240591, 5567398643], ['s', True, 1664763842.350336, 5567398643], ['t', True, 1664763842.541697, 5567398643], ['r', True, 1664763842.744139, 5567398643], ['o', True, 1664763842.8839538, 5567398643], ['n', True, 1664763843.0643358, 5567398643], ['g', True, 1664763843.225023, 5567398643], ['l', True, 1664763843.343898, 5567398643], ['y', True, 1664763843.589615, 5567398643], ['space', True, 1664763843.703682, 5567398643], ['w', True, 1664763843.992189, 5567398643], ['h', True, 1664763844.1348338, 5567398643], ['e', True, 1664763844.200282, 5567398643], ['r', True, 1664763844.318918, 5567398643], ['e', True, 1664763844.547151, 5567398643], ['s', False, 1664763844.7278311, 5567398643], ['a', True, 1664763844.994883, 5567398643], ['s', True, 1664763845.30838, 5567398643], ['space', True, 1664763845.491586, 5567398643], ['p', True, 1664763845.725918, 5567398643], ['r', True, 1664763845.9219441, 5567398643], ['e', True, 1664763846.031717, 5567398643], ['a', False, 1664763846.259287, 5567398643], ['s', True, 1664763846.355524, 5567398643], ['e', True, 1664763846.560896, 5567398643], ['n', True, 1664763846.791361, 5567398643], ['t', True, 1664763846.834503, 5567398643], ['i', False, 1664763847.1338181, 5567398643], ['a', True, 1664763847.415596, 5567398643], ['t', True, 1664763847.6451101, 5567398643], ['i', True, 1664763847.674866, 5567398643], ['o', True, 1664763847.841564, 5567398643], ['n', True, 1664763847.9723608, 5567398643], ['space', True, 1664763848.066498, 5567398643], ['m', True, 1664763848.253963, 5567398643], ['a', True, 1664763848.271243, 5567398643], ['t', False, 1664763848.5488582, 5567398643], ['k', False, 1664763848.930948, 5567398643], ['r', True, 1664763849.257448, 5567398643], ['k', True, 1664763849.509823, 5567398643], ['e', True, 1664763849.616588, 5567398643], ['t', True, 1664763849.827282, 5567398643], ['space', True, 1664763850.031171, 5567398643], ['e', True, 1664763850.230454, 5567398643], ['d', True, 1664763850.520572, 5567398643], ['u', True, 1664763850.795823, 5567398643], ['c', True, 1664763850.903883, 5567398643], ['a', True, 1664763851.063692, 5567398643], ['t', True, 1664763851.2720609, 5567398643], ['e', True, 1664763851.5380359, 5567398643], ['space', True, 1664763851.702581, 5567398643], ['s', True, 1664763851.944256, 5567398643], ['t', True, 1664763852.107628, 5567398643], ['r', True, 1664763852.3324459, 5567398643], ['a', True, 1664763852.618328, 5567398643], ['n', True, 1664763852.828981, 5567398643], ['g', True, 1664763853.030244, 5567398643], ['e', True, 1664763853.188457, 5567398643], ['space', True, 1664763853.283, 5567398643], ['t', True, 1664763853.489677, 5567398643], ['w', True, 1664763853.884048, 5567398643], ['i', True, 1664763854.11694, 5567398643], ['n', True, 1664763854.318987, 5567398643], ['space', True, 1664763854.499732, 5567398643], ['n', True, 1664763854.661432, 5567398643], ['a', True, 1664763854.7448251, 5567398643], ['k', True, 1664763854.960084, 5567398643], ['e', True, 1664763855.066377, 5567398643], ['space', False, 1664763855.213335, 5567398643], ['h', False, 1664763855.5460439, 5567398643], ['d', True, 1664763856.114646, 5567398643], ['space', True, 1664763856.2577019, 5567398643], ['h', True, 1664763856.4620092, 5567398643], ['e', True, 1664763856.526718, 5567398643], ['k', False, 1664763856.740629, 5567398643], ['l', True, 1664763857.075018, 5567398643], ['l', True, 1664763857.21296, 5567398643], ['o', True, 1664763857.4381282, 5567398643], ['space', True, 1664763857.569952, 5567398643], ['t', False, 1664763857.732262, 5567398643], ['u', True, 1664763857.89422, 5567398643], ['u', False, 1664763858.2061582, 5567398643], ['t', True, 1664763858.4502919, 5567398643], ['i', True, 1664763858.614638, 5567398643], ['l', True, 1664763858.863061, 5567398643], ['i', True, 1664763859.107553, 5567398643], ['t', True, 1664763859.3085032, 5567398643], ['y', True, 1664763859.4693592, 5567398643], ['space', True, 1664763859.608865, 5567398643], ['l', True, 1664763859.941081, 5567398643], ['i', True, 1664763860.121912, 5567398643], ['v', True, 1664763860.3001359, 5567398643], ['i', True, 1664763860.464122, 5567398643], ['n', True, 1664763860.622417, 5567398643], ['g', True, 1664763860.845618, 5567398643], ['space', True, 1664763860.915413, 5567398643], ['b', True, 1664763861.144715, 5567398643], ['l', True, 1664763861.3961082, 5567398643], ['o', True, 1664763861.555275, 5567398643], ['o', True, 1664763861.7160661, 5567398643], ['b', False, 1664763861.980082, 5567398643], ['d', True, 1664763862.397183, 5567398643], ['space', True, 1664763862.518532, 5567398643], ['b', True, 1664763862.829294, 5567398643], ['o', True, 1664763862.9942799, 5567398643], ['r', True, 1664763863.1486762, 5567398643], ['r', True, 1664763863.32976, 5567398643], ['o', True, 1664763863.452169, 5567398643], ['w', True, 1664763863.625121, 5567398643], ['space', True, 1664763863.764367, 5567398643], ['s', True, 1664763864.010347, 5567398643], ['t', True, 1664763864.1707132, 5567398643], ['i', True, 1664763864.447118, 5567398643], ['r', True, 1664763864.645206, 5567398643], ['space', True, 1664763864.7606962, 5567398643], ['p', True, 1664763865.082148, 5567398643], ['a', True, 1664763865.1604, 5567398643], ['t', False, 1664763865.9191282, 5567398643], ['e', False, 1664763866.077417, 5567398643], ['n', False, 1664763866.243031, 5567398643], ['r', True, 1664763866.507013, 5567398643], ['e', True, 1664763866.617735, 5567398643], ['n', True, 1664763866.6965199, 5567398643], ['t', True, 1664763866.832308, 5567398643], ['space', True, 1664763866.950073, 5567398643], ['p', True, 1664763867.141301, 5567398643], ['a', True, 1664763867.197227, 5567398643], ['r', True, 1664763867.300639, 5567398643], ['t', True, 1664763867.549254, 5567398643], ['i', True, 1664763867.666811, 5567398643], ['c', True, 1664763867.8469062, 5567398643], ['y', False, 1664763868.8786309, 5567398643], ['u', True, 1664763869.278211, 5567398643], ['l', True, 1664763869.4421592, 5567398643], ['a', True, 1664763869.4542038, 5567398643], ['r', True, 1664763869.576123, 5567398643]]

# sum(case when correct_key = 1 then 1 else 0 end) / ((max(time) - min(time)) / 60) / 5 wpm
count_correct_keys = len(list(filter(lambda x: x[1] == True, data)))
duration = data[-1][2] - data[0][2]
wpm = count_correct_keys / duration * 60 / 5
wpm

In [ ]:
query = '''
with tbl1 as (
select
    key
    , lead(key) over(partition by game_id order by game_id, time) following_key
    , time
    , lead(time) over(partition by game_id order by game_id, time) following_time
    , game_id

from keys_pressed
where 1=1
    and correct_key = 1
    and game_id > 100
)

select 
    following_key
    , avg(following_time - time) as time_diff
    , count(*) count

from tbl1
where 1=1
    and following_key is not null
group by 1
having 1=1
    and count > 100
order by time_diff asc
'''




# Function to read text file 
def load_query (query_name) -> list:
    '''Opens a given text file name and execute the 
    query to return a pd.DataFrame
    
    Parameter
    ---------
    query_name str: name of the file name to run
    '''

    # Open file and get the query 
    # with open(f'{current_dir}/data/queries/{query_name}') as file: 
    #     query = file.read()

    # Query database
    df = pd.read_sql_query(query, con)

    return df


# Load data
df_keytime = load_query('time_per_key_pressed.sql')

# Setting dictonary for each keys
key_score = dict(zip(df_keytime['following_key'], df_keytime['time_diff'].round(3)))
df_keytime


In [ ]:
query = '''
with tbl1 as (
select
    key
    , lead(key) over(partition by game_id order by game_id, time) following_key
    , time
    , lead(time) over(partition by game_id order by game_id, time) following_time
    , game_id

from keys_pressed
where 1=1
    and correct_key = 1
    and game_id > 100
)

select 
    following_key
    , game_id
    , max(time)
    , avg(following_time - time) as time_diff
    , count(*) count

from tbl1
where 1=1
    and following_key is not null
group by 2,1
having 1=1
    -- and count > 100
order by max(time) asc
'''



def load_query () -> list:
    '''Opens a given text file name and execute the 
    query to return a pd.DataFrame
    
    Parameter
    ---------
    query_name str: name of the file name to run
    '''

    # Open file and get the query 
    # with open(f'{current_dir}/data/queries/{query_name}') as file: 
    #     query = file.read()

    # Query database
    df = pd.read_sql_query(query, con)

    return df
 

df = load_query().reset_index()
df['index'] = df['index'] + 1
df

In [ ]:
px.line(df, x='index', y='time_diff')

In [ ]:
query = '''
    with tbl1 as (
    select
        key
        , lead(key) over(partition by game_id order by game_id, time) following_key
        , time
        , lead(time) over(partition by game_id order by game_id, time) following_time
        , game_id

    from keys_pressed
    where 1=1
        and correct_key = 1
        and game_id > 100
    )

    select 
        following_key
        , game_id
        , max(time)
        , avg(following_time - time) as time_diff
        , count(*) count

    from tbl1
    where 1=1
        and following_key is not null
    group by 2,1
    having 1=1
        -- and count > 100
    order by max(time) asc
    '''



def load_query () -> list:
    '''Opens a given text file name and execute the 
    query to return a pd.DataFrame
    
    Parameter
    ---------
    query_name str: name of the file name to run
    '''

    # Open file and get the query 
    # with open(f'{current_dir}/data/queries/{query_name}') as file: 
    #     query = file.read()

    # Query database
    df = pd.read_sql_query(query, con)

    return df
 

load_query ().query('following_key == "d"')

In [ ]:
print(pd.DataFrame([['Hugo', 20, 1, 1, 1], ['Julie', 24, 2, 2, 2], ['Nathan', 24, 2, 2, 3], ['Marie', 30, 4, 3, 4]], columns=['name', 'age', 'age_rank', 'age_dense_rank', 'age_row_number']).iloc[:, :2].to_markdown(index=False))

In [ ]:
from google.cloud import bigquery
# from google.oauth2 import service_account
# credentials = service_account.Credentials.from_service_account_file(
# 'path/to/file.json')

# project_id = 'my-bq'
client = bigquery.Client()

In [ ]:
import pandas as pd
import sqlite3
con = sqlite3.connect("database.db")


def push_to_gbq():
    df_keys_pressed = pd.read_sql_query('select * from keys_pressed', con)
    df_keys_pressed.to_gbq('pyfasttype.keys_pressed', if_exists='replace', progress_bar=None)
    df_clean_games_settings = pd.read_sql_query('select * from clean_games_settings', con)
    df_clean_games_settings['capitalized_letters_count_perc'] = df_clean_games_settings['capitalized_letters_count_perc'].astype(str)
    df_clean_games_settings.to_gbq('pyfasttype.clean_games_settings', if_exists='replace', progress_bar=None)

In [ ]:
files = ['database_afterbiggest.db', 'database_5.db', 'database_4.db', 'database.db']

con = sqlite3.connect('database.db')
pd.read_sql_query('select * from games_settings', con)


# df = pd.concat([df, pd.read_sql_query('select * from games_settings', con)], axis=0)

In [ ]:
# con = sqlite3.connect("main_database.db")


df = pd.DataFrame()


files = ['main_database_0.db', 'main_database.db']

for file in files:
    print(file)
    con = sqlite3.connect(file)


    current_df = pd.read_sql_query('select * from games_settings', con).drop_duplicates()

    if file == 'database_biggest.db': 
        cols = list(current_df.columns)
        # cols.remove('game_id')
        current_df['game_settings'] = current_df[cols].apply(lambda x: str({key:value for key, value in dict(x).items() if pd.notna(value)}), axis=1)
        current_df = current_df[['game_id', 'game_settings']]


    current_df = current_df[~current_df['game_settings'].str.contains('"{')]
    current_df = current_df[~current_df['game_settings'].str.contains("'{")]
    df = pd.concat([df, current_df], axis=0)



df_games_settings = df.drop_duplicates()
df_games_settings.to_sql('games_settings', con, if_exists='replace', index=False)
df_games_settings.shape

In [ ]:
con = sqlite3.connect("main_database.db")
df = pd.read_sql_query('select distinct * from games_settings', con)
df

In [ ]:
con = sqlite3.connect("main_database.db")
pd.read_sql_query('select distinct * from games_settings', con)

In [ ]:
def clean_games_settings():
    con = sqlite3.connect("main_database.db")
    df = pd.read_sql_query('select * from games_settings', con)
    print(df.shape)
    df['game_settings'] = df['game_settings'].apply(lambda x: eval(x))
    df_games_settings = pd.concat([df[['game_id']], df['game_settings'].apply(lambda x: pd.Series(x))], axis=1)
    df_games_settings.drop_duplicates().to_sql('clean_games_settings', con, if_exists='replace', index=False)

clean_games_settings()

In [ ]:
pd.Series({'sentence': 'hard', 'sentence_length': 1, 'max_word_length': None, 'capitalized_words_count': 0, 'capitalized_letters_count_perc': 0, 'punctuation_word_count_perc': 0, 'force_shift': False, 'hard_mode': False, 'train_letters': False, 'player_name': ' hehjksdhfkjhsd'})

In [ ]:
query = f"""
    select
        distinct
        kp.game_id
        , max(kp.time) datetime_unix
        , min(kp.time) mindatetime_unix
        , max(kp.time) - min(kp.time) game_duration
        , sum(case when kp.correct_key = 1 then 1 else 0 end) keys_to_press
        , count(*) keys_pressed
        , round(CAST(sum(case when kp.correct_key = 1 then 1 else 0 end) as REAL) / count(*), 3) accuracy
        , sum(case when kp.correct_key = 1 then 1 else 0 end) / ((max(kp.time) - min(kp.time)) / 60) / 5 wpm
        , length(cgs.sentence) sentence_lentgh
        , length(cgs.sentence) * (sum(case when kp.correct_key = 1 then 1 else 0 end) / ((max(kp.time) - min(kp.time)) / 60) / 5) * round(CAST(sum(case when kp.correct_key = 1 then 1 else 0 end) as REAL) / count(*), 3)score
        , cgs.max_word_length	
        , cgs.capitalized_words_count	
        , cgs.capitalized_letters_count_perc	
        , cgs.punctuation_word_count_perc	
        , cgs.force_shift	
        , cgs.hard_mode	
        , cgs.train_letters	
        , cgs.comment	
        , cgs.player_name

    from keys_pressed kp 
    left join clean_games_settings cgs using(game_id)
    where 1=1
        and cgs.game_id is not null
    group by 1
    """

    # query = 'select * from keys_pressed left join games_settings gs using(game_id)'

df_high_score = pd.read_sql_query(query, con)
df_high_score.sort_values('wpm', ascending=False).head(20)

In [ ]:

def whats_bestscore (sort_by='score', condition=''):
    query = f"""
        with tbl1 as (
        select
            game_id
            , max(time) - min(time) game_duration
            , sum(case when correct_key = 1 then 1 else 0 end) keys_to_press
            , count(*) keys_pressed
            , round(CAST(sum(case when correct_key = 1 then 1 else 0 end) as REAL) / count(*), 3) accuracy
            , sum(case when correct_key = 1 then 1 else 0 end) / ((max(time) - min(time)) / 60) / 5 wpm
            , length(cgs.sentence) sentence_lentgh
            , cgs.*

        from keys_pressed
        left join clean_games_settings cgs using(game_id)
        where 1=1
            --and game_id = 3513153090
        group by 1
        )

        select 
            *
        
        from tbl1
        where 1=1
            and keys_pressed > 100
            --wpm = (select max(wpm) from tbl1)
        """

    # query = 'select * from keys_pressed left join games_settings gs using(game_id)'

    df_high_score = pd.read_sql_query(query, con)
    df_high_score['score'] = (df_high_score['accuracy'] * df_high_score['wpm'].round(1) * df_high_score['sentence_lentgh']).round(0)
    df_high_score = df_high_score.sort_values(sort_by, ascending=False).reset_index(drop=True)
    best_player_name = df_high_score.loc[0, 'player_name']
    best_hard_mode = df_high_score.loc[0, 'hard_mode']
    best_game_duration = df_high_score.loc[0, 'game_duration']
    best_keys_to_press = df_high_score.loc[0, 'keys_to_press']
    best_keys_pressed = df_high_score.loc[0, 'keys_pressed']
    best_accuracy = df_high_score.loc[0, 'accuracy']
    best_wpm = df_high_score.loc[0, 'wpm']
    best_score = df_high_score.round().loc[0, 'score']
    return best_game_duration, best_keys_to_press, best_keys_pressed, best_accuracy, best_wpm, best_score, best_player_name, best_hard_mode

In [ ]:
def whats_bestscore (sort_by='score', condition=''):
    query = f"""
        select
            distinct
            kp.game_id
            , max(kp.time) datetime_unix
            , min(kp.time) mindatetime_unix
            , max(kp.time) - min(kp.time) game_duration
            , sum(case when kp.correct_key = 1 then 1 else 0 end) keys_to_press
            , count(*) keys_pressed
            , round(CAST(sum(case when kp.correct_key = 1 then 1 else 0 end) as REAL) / count(*), 3) accuracy
            , sum(case when kp.correct_key = 1 then 1 else 0 end) / ((max(kp.time) - min(kp.time)) / 60) / 5 wpm
            , length(cgs.sentence) sentence_lentgh
            , length(cgs.sentence) * (sum(case when kp.correct_key = 1 then 1 else 0 end) / ((max(kp.time) - min(kp.time)) / 60) / 5) * round(CAST(sum(case when kp.correct_key = 1 then 1 else 0 end) as REAL) / count(*), 3)score
            , cgs.max_word_length	
            , cgs.capitalized_words_count	
            , cgs.capitalized_letters_count_perc	
            , cgs.punctuation_word_count_perc	
            , cgs.force_shift	
            , cgs.hard_mode	
            , cgs.train_letters	
            , cgs.comment	
            , cgs.player_name

        from keys_pressed kp 
        left join clean_games_settings cgs using(game_id)
        where 1=1
            and cgs.game_id is not null
        group by 1
        having keys_to_press > 100
        """

        # query = 'select * from keys_pressed left join games_settings gs using(game_id)'

    df_high_score = pd.read_sql_query(query, con)   
    df_high_score.to_sql('summary_per_game', con, if_exists='replace', index=False)
    df_high_score = df_high_score.sort_values(sort_by, ascending=False).reset_index(drop=True)
    best_player_name = df_high_score.loc[0, 'player_name']
    best_hard_mode = df_high_score.loc[0, 'hard_mode']
    best_game_duration = df_high_score.loc[0, 'game_duration']
    best_keys_to_press = df_high_score.loc[0, 'keys_to_press']
    best_keys_pressed = df_high_score.loc[0, 'keys_pressed']
    best_accuracy = df_high_score.loc[0, 'accuracy']
    best_wpm = df_high_score.loc[0, 'wpm']
    best_score = df_high_score.round().loc[0, 'score']
    return best_game_duration, best_keys_to_press, best_keys_pressed, best_accuracy, best_wpm, best_score, best_player_name, best_hard_mode

whats_bestscore(sort_by='wpm')